# Temperatura de brilho TOA

Converte radiância térmica TOA em temperatura de brilho, com saída em **K**. A saída ainda não é a temperatura da superfície (LST).

## Precisa de

**Entradas para a conversão térmica:**

- **`thermal_band` — obrigatório:** imagem de radiância TOA da banda térmica, em **W m⁻² sr⁻¹ µm⁻¹**, na configuração padrão.
- **`band` — obrigatório:** número da banda, **sem unidade**; escolher **10 ou 11**.
- **`metadata` — utilizado neste exemplo:** metadados MTL da própria cena, **sem unidade**; fornecem as constantes de calibração da banda.

Como alternativa, passar a imagem original em **DN**, declarando `input_quantity="dn"`; com o MTL, a função converte para radiância antes de calcular a temperatura. Não passar Celsius ou LST como entrada. O fallback sem MTL está descrito na tabela e exige identificar corretamente o sensor e os coeficientes usados.

**Entrada opcional de qualidade:**

- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar, sujeito às validações do método. Neste exemplo, sua origem é descrita na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

O exemplo usa o leitor de cena **Landsat 8/9 Collection 2 Level 1**. Coloque na pasta da cena:

- **B4:** arquivo original do vermelho, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B5:** arquivo original do NIR, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B10:** arquivo térmico original, em **DN**; o leitor converte para BT10 em **K**.
- **B11:** arquivo térmico original, em **DN**; o leitor converte para BT11 em **K**.
- **MTL.txt:** metadados e coeficientes de calibração da própria cena; não é uma imagem.
- **QA_PIXEL:** códigos inteiros de qualidade, sem unidade física; o leitor interpreta os bits para montar a máscara.
- **QA_RADSAT:** códigos inteiros de saturação e oclusão de terreno, sem unidade física; complementam a máscara.

Essa lista corresponde ao leitor usado no notebook; uma chamada direta à função exige somente as entradas indicadas em “Precisa de”. B2 e SZA são opcionais para o leitor. Preserve os nomes originais e a grade dos arquivos. Não passar os arquivos QA brutos como máscara booleana.

## Como as imagens são preparadas

**TOA significa topo da atmosfera.** A grandeza também precisa ser especificada:

- **Radiância TOA:** energia medida pelo sensor, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Reflectância TOA:** **adimensional**, com ajuste do ângulo solar.
- **Temperatura de brilho TOA:** temperatura equivalente da radiância térmica, em **K**.

O leitor usa o **MTL da própria cena** para calibrar os arquivos originais. Não é necessário preparar manualmente as imagens TOA para executar este notebook.

A configuração `optical_quantity="radiance"` devolve as bandas ópticas em **radiância TOA, em W m⁻² sr⁻¹ µm⁻¹**; as bandas térmicas são devolvidas como temperaturas de brilho em **K**.

Trocar para `optical_quantity="reflectance"` muda a calibração óptica; isso exige verificar a compatibilidade com a parametrização usada no método. **Reflectância TOA ainda contém efeitos atmosféricos e é diferente de reflectância de superfície.**

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — banda térmica

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = brightness(
    # Banda térmica
    band=10,

    # Imagem de entrada e grandeza
    thermal_band=radiancia,
    input_quantity='radiance',

    # Calibração da própria cena
    metadata=cena.metadata,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `band` | Obrigatório | `Sem padrão` | Número da banda | 10 ou 11; aliases b10/b11 e band_10/band_11 também aceitos. Usar o MTL correspondente à banda térmica. |
| `thermal_band` | Obrigatório | `Sem padrão` | W m⁻² sr⁻¹ µm⁻¹ ou DN | Imagem térmica TOA em radiância por padrão; declarar input_quantity="dn" para DN. Mesma cena/banda do MTL. |
| `input_quantity` | Opcional | `"radiance"` | Seletor | "radiance" ou "dn"; declara a entrada. DN é convertido com os coeficientes da mesma banda no MTL. |
| `metadata` | Obrigatório na rota deste exemplo | `None` | Dicionário MTL | MTL da mesma cena; fornece K1/K2 e, para DN, ganho/offset. O arquivo é lido por LandsatScene/read_mtl. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — entrada em DN com MTL

DN da banda selecionada; a função aplica ganho e offset do MTL antes da conversão térmica.

```python
resultado = brightness(
    # Banda térmica
    band=10,

    # Imagem de entrada e grandeza
    thermal_band=dn,
    input_quantity='dn',

    # Calibração da própria cena
    metadata=cena.metadata,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `band` | Obrigatório | `Sem padrão` | Número da banda | 10 ou 11; aliases b10/b11 e band_10/band_11 também aceitos. Usar o MTL correspondente à banda térmica. |
| `thermal_band` | Obrigatório | `Sem padrão` | W m⁻² sr⁻¹ µm⁻¹ ou DN | Imagem térmica TOA em radiância por padrão; declarar input_quantity="dn" para DN. Mesma cena/banda do MTL. |
| `input_quantity` | Opcional | `"radiance"` | Seletor | "radiance" ou "dn"; declara a entrada. DN é convertido com os coeficientes da mesma banda no MTL. |
| `metadata` | Obrigatório na rota deste exemplo | `None` | Dicionário MTL | MTL da mesma cena; fornece K1/K2 e, para DN, ganho/offset. O arquivo é lido por LandsatScene/read_mtl. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — fallback sem MTL

Preferir o MTL. Sem metadata, declarar sensor e fornecer ganho/offset da própria cena para DN. Com entrada em radiância, omitir rad_gain/rad_bias. A saída continua sendo BT em K.

```python
resultado = brightness(
    # Banda térmica
    band=10,

    # Imagem de entrada e grandeza
    thermal_band=dn,
    input_quantity='dn',

    # Calibração da própria cena
    sensor='landsat_8',
    rad_gain=ganho_da_cena,
    rad_bias=offset_da_cena,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `band` | Obrigatório | `Sem padrão` | Número da banda | 10 ou 11; aliases b10/b11 e band_10/band_11 também aceitos. Usar o MTL correspondente à banda térmica. |
| `thermal_band` | Obrigatório | `Sem padrão` | W m⁻² sr⁻¹ µm⁻¹ ou DN | Imagem térmica TOA em radiância por padrão; declarar input_quantity="dn" para DN. Mesma cena/banda do MTL. |
| `input_quantity` | Opcional | `"radiance"` | Seletor | "radiance" ou "dn"; declara a entrada. DN é convertido com os coeficientes da mesma banda no MTL. |
| `sensor` | Condicional: fallback sem MTL | `None → landsat_8` | Seletor | "landsat_8" ou "landsat_9"; usa constantes térmicas de fallback. Preferir metadata da própria cena; omitir no fluxo com MTL. |
| `rad_gain` | Opcional no fallback DN; recomendado informar | `None → constante do sensor` | Radiância por DN | Ganho radiométrico da banda, obtido na própria cena; não reutilizar coeficientes de outra cena. |
| `rad_bias` | Opcional no fallback DN; recomendado informar | `None → constante do sensor` | W m⁻² sr⁻¹ µm⁻¹ | Offset radiométrico da mesma banda e cena. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import brightness, toa_radiance
from pylstemp.scene import LandsatScene

pasta = Path("CAMINHO_DA_CENA")
saida = Path("resultados/brightness")
saida.mkdir(parents=True, exist_ok=True)
banda = 10
entrada = "radiance"  # radiance ou dn

## Ler e calibrar a imagem

`LandsatScene` confere os arquivos/grade da cena e le o MTL. `calibrated()` retorna bandas opticas, BT10/BT11 em K e mascara QA booleana. As entradas deste exemplo sao radiancias opticas TOA.

### Máscara usada neste exemplo

`mask` é uma máscara **de exclusão**, não uma máscara de seleção: `True` rejeita o pixel e `False` permite o cálculo. O leitor usa QA_PIXEL para fill, nuvem dilatada, cirrus de alta confiança, nuvem, sombra e neve; QA_RADSAT para saturação óptica B2/B4/B5 e oclusão de terreno; e verifica SZA quando disponível. A opção `exclude_water=True` também exclui água (bit7 de QA_PIXEL). Valores não finitos ou fora do domínio podem ser rejeitados adicionalmente pelo cálculo. `mask=None` em uma chamada direta não cria automaticamente uma máscara de nuvens.


In [ ]:
cena = LandsatScene(pasta)
opticas, bt, mascara = cena.calibrated(optical_quantity="radiance")

## Aplicar o metodo

Use os parametros da propria cena e respeite o dominio e o sensor da fonte.

In [ ]:
dn = cena.read('B' + str(banda))
if entrada == 'radiance':
    radiancia = toa_radiance(dn, metadata=cena.metadata, band=banda, mask=mascara)
    imagem = brightness(
        # Banda térmica
        band=banda,

        # Imagem de entrada e grandeza
        thermal_band=radiancia,
        input_quantity='radiance',

        # Calibração da própria cena
        metadata=cena.metadata,

        # Exclusão de pixels
        mask=mascara,
    )
else:
    imagem = brightness(
        # Banda térmica
        band=banda,

        # Imagem de entrada e grandeza
        thermal_band=dn,
        input_quantity='dn',

        # Calibração da própria cena
        metadata=cena.metadata,

        # Exclusão de pixels
        mask=mascara,
    )


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
cena.write_geotiff(saida / "brightness.tif", imagem, units='K')
cena.close()
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

[Artigo ou documentacao original](https://www.usgs.gov/landsat-missions/using-usgs-landsat-level-1-data-product).

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).